# Studio 2 — Kelompok 06 · Training di Kaggle

Sebelum menjalankan:
1. **Add Input** → pilih dataset Kaggle *private* berisi folder `train/` (5 folder kelas).
2. **Settings → Accelerator**: `GPU T4 x2`.
3. **Settings → Internet**: ON (untuk `git clone` dan mengunduh bobot pretrained).

Setelah selesai, unduh dari panel **Output** (`/kaggle/working/`):
- `model.pth` → taruh di `submission/model.pth` repo lokal
- `experiments/*.txt` → taruh di folder `experiments/` repo lokal

In [ ]:
!git clone https://github.com/beningapniprameswari/Studio-2_Kelompok-6.git
%cd /kaggle/working/Studio-2_Kelompok-6
!pip install -q -r submission/requirements.txt
!mkdir -p experiments
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Cari lokasi folder train/ di input Kaggle
import glob
cands = [p for p in glob.glob('/kaggle/input/**/train', recursive=True) if glob.glob(p + '/botol_plastik')]
DATA = cands[0]
print(DATA)

## 1. Baseline ResNet-18 (pembanding, opsional)
Bobot disimpan ke `baseline/model.pth`, tidak menimpa submission utama.

In [ ]:
!python train_baseline.py --data {DATA} --epochs 8 2>&1 | tee experiments/01_baseline_resnet18.txt
!python evaluate.py --submission ./baseline --data ./my_val 2>&1 | tee -a experiments/01_baseline_resnet18.txt

## 2. ConvNeXt-Tiny (submission utama)
Latih di 85% data, pilih epoch terbaik dengan Macro-F1 `my_val/` (split yang sama dengan baseline).

In [ ]:
!python train.py --data {DATA} --epochs 15 2>&1 | tee experiments/02_convnext_tiny.txt

In [ ]:
# Uji submission persis seperti asisten (evaluate.py resmi)
!python evaluate.py --submission ./submission --data ./my_val 2>&1 | tee -a experiments/02_convnext_tiny.txt
!ls -lh submission/model.pth
!cp submission/model.pth /kaggle/working/model.pth
!cp -r experiments /kaggle/working/